# Stage 1: Experimental — VWAP-based wallet preselection

Preselect larger wallet sets that are profitable (average_roi > 0.02), then
compute per-trade trailing 15-minute VWAP and VWAP volume for BUY trades
by (wallet, condition_id, token_id).

These VWAP features are added back to the trade DataFrames for downstream
analysis.

**Output:** `stage1_experimental_result.json` with preselected wallets + VWAP stats.

In [379]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from IPython.display import display

from lib import (
    load_trades,
    compute_copyable_notional,
    compute_opening_metrics,
    DEFAULT_TAGS,
)
from polymarket_analysis.wallet_selection.volatility import compute_wallet_metrics


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Parameters

In [380]:
MIN_ROI = 0.03
VWAP_WINDOW_MINUTES = 15

print(f"MIN_ROI: {MIN_ROI}")
print(f"VWAP_WINDOW_MINUTES: {VWAP_WINDOW_MINUTES}")

MIN_ROI: 0.03
VWAP_WINDOW_MINUTES: 15


## Load data

In [381]:
df_full = load_trades()
df_full = compute_copyable_notional(df_full)

train_cutoff = pd.Timestamp("2026-06-01", tz="UTC")
val_cutoff = pd.Timestamp("2026-07-01", tz="UTC")

df_train = df_full[df_full["dt"] < train_cutoff].copy()
df_val = df_full[(df_full["dt"] >= train_cutoff) & (df_full["dt"] < val_cutoff)].copy()
df_test = df_full[df_full["dt"] >= val_cutoff].copy()

print(f"Split by trade date:")
print(f"  Train: {len(df_train):>10,} trades  ({df_train['condition_id'].nunique():>5,} markets)  < {train_cutoff.date()}")
print(f"  Val:   {len(df_val):>10,} trades  ({df_val['condition_id'].nunique():>5,} markets)  {train_cutoff.date()} .. {val_cutoff.date()}")
print(f"  Test:  {len(df_test):>10,} trades  ({df_test['condition_id'].nunique():>5,} markets)  >= {val_cutoff.date()}")
print(f"  Total: {len(df_full):>10,} trades  ({df_full['condition_id'].nunique():>5,} markets)")

# Market overlap check
train_markets = set(df_train["condition_id"].unique())
val_markets = set(df_val["condition_id"].unique())
test_markets = set(df_test["condition_id"].unique())
print(f"\n  Markets overlapping train/val: {len(train_markets & val_markets)}")
print(f"  Markets overlapping train/test: {len(train_markets & test_markets)}")
print(f"  Markets overlapping val/test: {len(val_markets & test_markets)}")

Markets: 1974837
Filtered markets for {'Weather'}: 91123
Loading 16 trade shards...
Total trades loaded: 14,250,603
Unique wallets: 4,082
Date range: 2025-01-09 15:32:39+00:00 -> 2026-07-27 06:12:25+00:00
Split by trade date:
  Train:  6,035,492 trades  (23,237 markets)  < 2026-06-01
  Val:    4,766,255 trades  (19,787 markets)  2026-06-01 .. 2026-07-01
  Test:   3,448,856 trades  (16,333 markets)  >= 2026-07-01
  Total: 14,250,603 trades  (56,875 markets)

  Markets overlapping train/val: 1350
  Markets overlapping train/test: 1
  Markets overlapping val/test: 1132


## Compute wallet metrics on training data

In [382]:
wallet_vol, _ = compute_wallet_metrics(df_train)

wallet_vol["copyable_pnl_factor"] = np.clip(
    wallet_vol["copyable_pnl"] / wallet_vol["total_pnl"].replace(0, np.nan),
    0, 1.0,
).fillna(0.0)
wallet_vol["copyable_roi"] = wallet_vol["average_roi"] * wallet_vol["copyable_pnl_factor"]

opening_metrics = compute_opening_metrics(df_train)
wallet_vol = wallet_vol.merge(opening_metrics, on="wallet", how="left")
for c in ["opening_roi", "opening_pnl", "opening_copyable_roi", "opening_copyable_pnl"]:
    wallet_vol[c] = wallet_vol[c].fillna(0.0)

print(f"Wallets with metrics: {len(wallet_vol)}")
wallet_vol[["wallet", "buy_roi", "sell_roi", "copyable_pnl", "copyable_roi", "num_buckets"]].head(10)

Wallets with metrics: 3584


,wallet,buy_roi,sell_roi,copyable_pnl,copyable_roi,num_buckets
0,0x00546dfeb4097e232c86a775ccbe3c9b84c0cab1,0.023563,NaN,12.845816,0.037440,53
1,0x0054ee7dfb882d2d016fa13ef5f5cdb3b0ebcf1f,0.005054,0.002003,-3.331733,0.000000,235
2,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0.026175,0.036027,227.315103,0.011217,5511
3,0x00833cc2d777e6f2fc8437679124024ae6468cb1,NaN,NaN,0.000000,NaN,2
4,0x0141be702d272f17666e280303ad44e7bc0cc2da,0.677602,-1.000000,119.386767,0.504015,50
5,0x015be8bad14c79d2722a0bd8bbe0cd93b905556d,NaN,NaN,-20.495392,NaN,344
6,0x01a5fb1fa13f378138a31382c8364b5d4e2b0e36,0.012020,-1.000000,12.681270,-0.101257,81
7,0x01a68281185e728ba0fef6245008bf8af68a59b0,0.025917,-0.181527,136.243576,0.005965,5369
8,0x01ced860d8dca5d7987579d2a2635df8520d27a2,0.196781,-0.063492,-316.535035,0.000000,1985
9,0x01d94480e2a96cdd01fed071878b1adf82e0acd0,0.005820,-0.969739,24.241672,0.005206,2999


## Preselect wallets by average buy ROI

In [383]:
wallet_vol.columns

Index(['wallet', 'pnl_volatility', 'num_buckets', 'num_markets', 'trade_count',
       'total_notional', 'total_pnl', 'copyable_pnl', 'top5_pnl_pct',
       'top10_pnl_pct', 'worst5_pnl_pct', 'top_market_pnl_pct',
       'top_market_abs_pnl_pct', 'market_pnl_hhi', 'positive_bucket_share',
       'median_roi', 'median_dt', 'average_roi', 'buy_roi', 'buy_pnl',
       'buy_notional', 'buy_copyable_pnl', 'sell_pnl', 'sell_roi',
       'max_drawdown', 'max_drawdown_to_pnl', 'max_copyable_drawdown',
       'max_copyable_drawdown_to_copyable_pnl', 'return',
       'copyable_pnl_factor', 'copyable_roi', 'opening_pnl',
       'opening_notional', 'opening_copyable_pnl', 'opening_buys',
       'opening_copyable_notional', 'opening_roi', 'opening_copyable_roi'],
      dtype='object')

In [384]:
preselected_ws = set(
    wallet_vol.loc[wallet_vol["buy_roi"] > MIN_ROI, "wallet"]
)
print(f"Preselected wallets (buy_roi > {MIN_ROI}): {len(preselected_ws)}")

# Quick stats on the preselected set
preselected_df = wallet_vol[wallet_vol["wallet"].isin(preselected_ws)].copy()
print()
print(f"  Buy ROI range:  {preselected_df['buy_roi'].min():.4f} — {preselected_df['buy_roi'].max():.4f}")
print(f"  Avg num_buckets: {preselected_df['num_buckets'].mean():.0f}")
print(f"  total_pnl: ${preselected_df['total_pnl'].sum():,.0f}")
print(f"  trades: {preselected_df['trade_count'].sum():,.0f}")

Preselected wallets (buy_roi > 0.03): 23

  Buy ROI range:  0.0309 — 1.5416
  Avg num_buckets: 28516
  total_pnl: $102,669
  Buy copyable PnL: $-36,250
  trades: 738,994


In [385]:
wallet_vol.columns

Index(['wallet', 'pnl_volatility', 'num_buckets', 'num_markets', 'trade_count',
       'total_notional', 'total_pnl', 'copyable_pnl', 'top5_pnl_pct',
       'top10_pnl_pct', 'worst5_pnl_pct', 'top_market_pnl_pct',
       'top_market_abs_pnl_pct', 'market_pnl_hhi', 'positive_bucket_share',
       'median_roi', 'median_dt', 'average_roi', 'buy_roi', 'buy_pnl',
       'buy_notional', 'buy_copyable_pnl', 'sell_pnl', 'sell_roi',
       'max_drawdown', 'max_drawdown_to_pnl', 'max_copyable_drawdown',
       'max_copyable_drawdown_to_copyable_pnl', 'return',
       'copyable_pnl_factor', 'copyable_roi', 'opening_pnl',
       'opening_notional', 'opening_copyable_pnl', 'opening_buys',
       'opening_copyable_notional', 'opening_roi', 'opening_copyable_roi'],
      dtype='object')

## Compute 15-min trailing VWAP for BUY trades

For each BUY trade by a preselected wallet, look back 15 minutes over
the same (wallet, condition_id, token_id) and compute:
- **vwap_15m**: volume-weighted average price of trades **strictly before** this one
- **vwap_volume_15m**: total USDC volume of trades in the window

> `TEST_MODE=True` limits to 50 wallets for quick validation.

In [386]:
# buy_mask = df_full["wallet"].isin(preselected_ws) & (df_full["side"] == "BUY") 
# buy_trades = df_full.loc[buy_mask].copy() 
# print(f"BUY trades by preselected wallets: {len(buy_trades):,}")

# buy_trades = buy_trades.sort_values(
#     ["wallet", "condition_id", "token_id", "dt"],
#     kind="mergesort"
# ).reset_index(drop=True)

# buy_trades["vwap_15m"] = np.nan
# buy_trades["vwap_volume_15m"] = 0.0

# window_ns = np.timedelta64(VWAP_WINDOW_MINUTES, "m")

# for _, idx in buy_trades.groupby(
#     ["wallet", "condition_id", "token_id"],
#     sort=False
# ).groups.items():

#     g = buy_trades.loc[idx]

#     t = g["dt"].values
#     qty = g["quantity"].to_numpy()
#     usdc = g["usdc_amount"].to_numpy()
#     pq = (g["price"] * g["quantity"]).to_numpy()

#     left = 0
#     sum_qty = 0.0
#     sum_pq = 0.0
#     sum_usdc = 0.0

#     out_vwap = np.empty(len(g))
#     out_vol = np.empty(len(g))

#     for i in range(len(g)):

#         # remove expired trades
#         while left < i and t[left] < t[i] - window_ns:
#             sum_qty -= qty[left]
#             sum_pq -= pq[left]
#             sum_usdc -= usdc[left]
#             left += 1

#         # current trade is excluded
#         out_vwap[i] = np.nan if sum_qty == 0 else sum_pq / sum_qty
#         out_vol[i] = sum_usdc

#         # add current trade
#         sum_qty += qty[i]
#         sum_pq += pq[i]
#         sum_usdc += usdc[i]

#     buy_trades.loc[idx, "vwap_15m"] = out_vwap
#     buy_trades.loc[idx, "vwap_volume_15m"] = out_vol

# df_full = df_full.merge(
#     buy_trades[["tx_hash","vwap_15m","vwap_volume_15m"]],
#     on="tx_hash",
#     how="left",
# )

# df_train = df_full[df_full["dt"] < train_cutoff].copy()
# df_val = df_full[(df_full["dt"] >= train_cutoff) & (df_full["dt"] < val_cutoff)].copy()
# df_test = df_full[df_full["dt"] >= val_cutoff].copy()

In [387]:
wallet_vol.columns

Index(['wallet', 'pnl_volatility', 'num_buckets', 'num_markets', 'trade_count',
       'total_notional', 'total_pnl', 'copyable_pnl', 'top5_pnl_pct',
       'top10_pnl_pct', 'worst5_pnl_pct', 'top_market_pnl_pct',
       'top_market_abs_pnl_pct', 'market_pnl_hhi', 'positive_bucket_share',
       'median_roi', 'median_dt', 'average_roi', 'buy_roi', 'buy_pnl',
       'buy_notional', 'buy_copyable_pnl', 'sell_pnl', 'sell_roi',
       'max_drawdown', 'max_drawdown_to_pnl', 'max_copyable_drawdown',
       'max_copyable_drawdown_to_copyable_pnl', 'return',
       'copyable_pnl_factor', 'copyable_roi', 'opening_pnl',
       'opening_notional', 'opening_copyable_pnl', 'opening_buys',
       'opening_copyable_notional', 'opening_roi', 'opening_copyable_roi'],
      dtype='object')

In [388]:
bad_buy_leaders = wallet_vol[
    (wallet_vol['buy_roi'] >= 0.04)
    & (wallet_vol['trade_count'] >= 1000)
    & (wallet_vol['total_pnl'] > 10000)
    & (wallet_vol['max_drawdown_to_pnl'] <= 0.5)
    & (wallet_vol['copyable_roi'] < 0.02)
    & (wallet_vol['buy_copyable_pnl'] * -1 >= 1000)
]

print(f"Bad buy leaders: {len(bad_buy_leaders)}")
print(f"Bad buy leader train trades: {len(df_train[df_train['wallet'].isin(bad_buy_leaders['wallet'])])}")
print(f"Bad buy leader pnl: ${bad_buy_leaders['total_pnl'].sum():,.0f}")
print(f"Bad buy leader copyable buy pnl: ${bad_buy_leaders['buy_copyable_pnl'].sum():,.0f}")
print(f"Bad buy leader buy roi: ${bad_buy_leaders['buy_pnl'].sum() / bad_buy_leaders['buy_notional'].sum():,.2f}")
print(f"Bad buy leader val pnl: ${df_val[df_val['wallet'].isin(bad_buy_leaders['wallet'])]['pnl'].sum():,.0f}")

Bad buy leaders: 2
Bad buy leader train trades: 133447
Bad buy leader pnl: $29,390
Bad buy leader copyable buy pnl: $-11,750
Bad buy leader buy roi: $0.07
Bad buy leader val pnl: $9,746


In [389]:
bad_buy_leaders.head()

,wallet,pnl_volatility,num_buckets,num_markets,trade_count,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top10_pnl_pct,...,return,copyable_pnl_factor,copyable_roi,opening_pnl,opening_notional,opening_copyable_pnl,opening_buys,opening_copyable_notional,opening_roi,opening_copyable_roi
1153,0xb06a0eae498750ed0acac7e1f759f741c56e52f5,0.322946,92384,10339,103599,263354.778806,15099.549014,-9365.720930,0.275840,0.435316,...,0.057335,0.0,0.0,1351.187818,17103.279037,-271.397678,10298.0,6479.346074,0.079002,-0.041887
1289,0xc7d02944a76b9f83b199e9090ecc92c82d241f8a,0.434007,28639,4309,29848,165292.295427,14290.413849,-3584.065829,0.300944,0.409550,...,0.086455,0.0,0.0,3230.185197,43866.304970,-1175.765783,5857.0,10039.604894,0.073637,-0.117113


In [433]:
wallet_vol.columns

Index(['wallet', 'pnl_volatility', 'num_buckets', 'num_markets', 'trade_count',
       'total_notional', 'total_pnl', 'copyable_pnl', 'top5_pnl_pct',
       'top10_pnl_pct', 'worst5_pnl_pct', 'top_market_pnl_pct',
       'top_market_abs_pnl_pct', 'market_pnl_hhi', 'positive_bucket_share',
       'median_roi', 'median_dt', 'average_roi', 'buy_roi', 'buy_pnl',
       'buy_notional', 'buy_copyable_pnl', 'sell_pnl', 'sell_roi',
       'max_drawdown', 'max_drawdown_to_pnl', 'max_copyable_drawdown',
       'max_copyable_drawdown_to_copyable_pnl', 'return',
       'copyable_pnl_factor', 'copyable_roi', 'opening_pnl',
       'opening_notional', 'opening_copyable_pnl', 'opening_buys',
       'opening_copyable_notional', 'opening_roi', 'opening_copyable_roi'],
      dtype='object')

In [439]:
(
    wallet_vol[
        (wallet_vol['trade_count'] >= 500)
        # & (wallet_vol['average_roi'] >= 0.05)
        # & (wallet_vol['copyable_pnl'] >= 100)
        ]
    ).sort_values('total_pnl', ascending=False).head(20)

,wallet,pnl_volatility,num_buckets,num_markets,trade_count,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top10_pnl_pct,...,return,copyable_pnl_factor,copyable_roi,opening_pnl,opening_notional,opening_copyable_pnl,opening_buys,opening_copyable_notional,opening_roi,opening_copyable_roi
1149,0xafde461fce5aa0fabdb7711c59db93b65e343e1d,0.957756,3163,394,3718,1.092031e+05,30043.942187,11841.913827,1.032128,1.162751,...,0.275120,0.394153,0.487646,3969.194801,6349.977287,513.202781,446.0,1857.731022,0.625072,0.276252
1143,0xae2e04fe9d8ccba5e45ba17ddf9dfbef498c40ad,1.857969,3859,484,4716,1.033217e+05,22667.525475,2248.495040,1.158076,1.258107,...,0.219388,0.099195,0.065497,4883.986659,10320.547633,-913.980441,580.0,3223.326167,0.473229,-0.283552
1173,0xb40e89677d59665d5188541ad860450a6e2a7cc9,0.097495,266467,7853,286565,1.211482e+06,18046.306392,-23867.685176,0.106982,0.180791,...,0.014896,0.000000,0.000000,2189.775995,210062.939829,-3181.590138,46879.0,82367.297536,0.010424,-0.038627
498,0x488c725253fc21c7a9ca812030dc2f6343f98c1c,1.609136,2359,391,3269,1.861033e+05,16867.802367,2569.051445,0.732761,0.990217,...,0.090637,0.152305,0.089965,8224.460232,25504.823979,371.075386,475.0,3683.864651,0.322467,0.100730
1153,0xb06a0eae498750ed0acac7e1f759f741c56e52f5,0.322946,92384,10339,103599,2.633548e+05,15099.549014,-9365.720930,0.275840,0.435316,...,0.057335,0.000000,0.000000,1351.187818,17103.279037,-271.397678,10298.0,6479.346074,0.079002,-0.041887
1289,0xc7d02944a76b9f83b199e9090ecc92c82d241f8a,0.434007,28639,4309,29848,1.652923e+05,14290.413849,-3584.065829,0.300944,0.409550,...,0.086455,0.000000,0.000000,3230.185197,43866.304970,-1175.765783,5857.0,10039.604894,0.073637,-0.117113
947,0x8fb431f5057112cfdb0e7f566b4b687cb69cb9ed,0.786399,3379,358,4248,8.515574e+04,12334.921901,3913.186756,0.431021,0.664747,...,0.144851,0.317245,0.035503,2281.637325,12308.219513,1063.131160,413.0,3591.994193,0.185375,0.295972
927,0x8d0930676d559cc8fb7d8af0c555791c1820143f,0.112830,5356,697,5896,1.150638e+06,11769.118661,2002.084483,0.261282,0.403760,...,0.010228,0.170113,0.089243,4130.966471,245683.787199,849.066007,711.0,10868.312838,0.016814,0.078123
1585,0xf1e18ec32b2f1e123bc098e3956e6fd00012c152,1.729986,1437,155,2745,3.164178e+04,11222.055485,2332.129447,0.696916,0.872260,...,0.354659,0.207817,0.436024,1971.908374,2595.435486,436.082675,166.0,640.238506,0.759760,0.681125
1410,0xdafdc201e4a769c4424462f9210763b221da2ad4,2.106591,1691,278,2174,9.428722e+04,10559.840084,943.980646,1.078021,1.280717,...,0.111997,0.089393,0.004235,2816.186397,27036.603046,856.983938,290.0,3244.420900,0.104162,0.264141


In [400]:
if 'bad_leader_wallet' not in df_full.columns:
    bad_buy_leader_trades = df_full[(df_full['wallet'].isin(bad_buy_leaders['wallet'])) & (df_full['side'] == 'BUY')]
    print(f"Bad buy leader full trades: {len(bad_buy_leader_trades)}")

    leaders = bad_buy_leader_trades.rename(columns={"dt": "dt_leader", "wallet": "bad_leader_wallet"})[['dt_leader', 'bad_leader_wallet', 'condition_id', 'outcome']]

    df_full = pd.merge_asof(
        df_full.sort_values("dt"),
        leaders.sort_values("dt_leader"),
        left_on="dt",
        right_on="dt_leader",
        by=["condition_id", "outcome"],
        direction="backward",
        tolerance=pd.Timedelta(minutes=5),
        allow_exact_matches=False,
)

In [401]:
df_train = df_full[df_full["dt"] < train_cutoff].copy()
df_val = df_full[(df_full["dt"] >= train_cutoff) & (df_full["dt"] < val_cutoff)].copy()
df_test = df_full[df_full["dt"] >= val_cutoff].copy()

print(f"Bad leader trades in train: {len(df_train[df_train['bad_leader_wallet'].notnull()])}")
print(f"Bad leader trades in val: {len(df_val[df_val['bad_leader_wallet'].notnull()])}")
print(f"Bad leader trades in test: {len(df_test[df_test['bad_leader_wallet'].notnull()])}")

Bad leader trades in train: 310208
Bad leader trades in val: 114080
Bad leader trades in test: 53493


In [402]:
candidates = wallet_vol.copy()

base_mask = (
    (candidates['buy_roi'] >= 0.05)
    # & (candidates['median_roi'] >= 0)
    & (candidates['num_buckets'] >= 20)
    & (candidates['num_markets'] >= 15)
    & (candidates['max_drawdown_to_pnl'] <= 0.2)
    & (candidates['copyable_roi'] >= 0.05)
    # & (candidates['top_market_pnl_pct'] < 0.25)
    # & (candidates['top5_pnl_pct'] < 0.55)
    # & (candidates['top10_pnl_pct'].fillna(0.85) < 0.8)
    # & (candidates['top_market_abs_pnl_pct'].fillna(candidates['top_market_pnl_pct']) < 0.25)
    & (candidates['market_pnl_hhi'].fillna(0.20) < 0.20)
    & (candidates['median_dt'].dt.date <= (pd.Timestamp.today().date() - pd.Timedelta(days=30)))
    & (candidates['total_notional'] >= 1_000)
)
candidates = candidates[base_mask]
print(f"Candidate wallets after base mask: {len(candidates)}")

Candidate wallets after base mask: 39


In [403]:
candidate_trades = df_test[df_test["wallet"].isin(candidates["wallet"])].copy()

In [405]:
len(candidate_trades), len(candidate_trades[candidate_trades["bad_leader_wallet"].notnull()]), len(candidate_trades[candidate_trades["bad_leader_wallet"].isnull()])

(70918, 1225, 69693)

In [415]:

# def print_pnl()

(
    candidate_trades[
        (candidate_trades['side'] == 'BUY')
        # & (candidate_trades['position'] == candidate_trades['quantity'])
        & (candidate_trades['bad_leader_wallet'].isna())
        # & (candidate_trades["vwap_volume_15m"].isna())

        #  (
        #     (candidate_trades["vwap_15m"] - 0.01 < candidate_trades["price"])
        #     & (candidate_trades["vwap_15m"] + 0.01 > candidate_trades["price"])
        #     & (candidate_trades["vwap_volume_15m"] < 100)
        # )
    ]
.groupby('side').agg(
    trades=('wallet', 'count'),
    pnl=('pnl', 'sum'),
    copyable_pnl=('copyable_pnl', 'sum'),
    copyable_notional=('copyable_notional', 'sum'),
    notional=('notional', 'sum'),
).assign(
    copyable_roi=lambda x: x['copyable_pnl'] / x['copyable_notional'].replace(0, np.nan),
    roi=lambda x: x['pnl'] / x['notional'].replace(0, np.nan),
).sort_values('copyable_pnl', key=abs, ascending=False)
)

,trades,pnl,copyable_pnl,copyable_notional,notional,copyable_roi,roi
side,,,,,,,
BUY,56516,67412.97403,10579.908325,114419.961201,627082.73091,0.092466,0.107503


## Save stage 1 experimental result

In [398]:
import json
from datetime import datetime, timezone
from pathlib import Path


def _convert(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


# VWAP stats per wallet
buy_vwap_stats = (
    buy_trades.groupby("wallet", sort=False)["vwap_15m"]
    .agg(["mean", "median", "std", "count"])
    .rename(columns={"mean": "vwap_mean", "median": "vwap_median",
                     "std": "vwap_std", "count": "vwap_trades"})
    .reset_index()
)

wallet_cols = [
    "wallet", "buy_roi", "sell_roi", "average_roi", "copyable_pnl",
    "num_buckets", "num_markets", "total_notional", "total_pnl",
    "market_pnl_hhi",
]
cols_present = [c for c in wallet_cols if c in wallet_vol.columns]
wallet_records = (
    wallet_vol[wallet_vol["wallet"].isin(preselected_ws)][cols_present]
    .merge(buy_vwap_stats, on="wallet", how="left")
    .to_dict(orient="records")
)
wallet_records = [{k: _convert(v) for k, v in w.items()} for w in wallet_records]

vwap_fill_rate = float(df_full["vwap_15m"].notna().mean())

metadata = {
    "type": "experimental",
    "tags": sorted(DEFAULT_TAGS),
    "run_timestamp": datetime.now(timezone.utc).isoformat(),
    "n_preselected_wallets": len(preselected_ws),
    "n_wallets_total": len(wallet_vol),
    "vwap_fill_rate": vwap_fill_rate,
    "vwap_window_minutes": VWAP_WINDOW_MINUTES,
    "min_roi": MIN_ROI,
}

payload = {
    "stage": 1,
    "type": "experimental",
    "metadata": metadata,
    "wallets": wallet_records,
}

out_path = Path("stage1_experimental_result.json")
with open(out_path, "w") as f:
    json.dump(payload, f, indent=2)
print(f"Saved stage 1 experimental result -> {out_path.resolve()}")

KeyError: 'vwap_15m'